In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.preprocessing import MinMaxScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    mean_absolute_error, r2_score
)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

RANDOM_STATE = 42

df = pd.read_csv("world_cup_dataset.csv")
print(df.head())


        team confederation  fifa_ranking  avg_goals_scored_per_game  avg_goals_conceded_per_game  win_rate_last_20  draw_rate_last_20  \
0     Brazil      CONMEBOL             4                        2.3                          0.8              0.75               0.15   
1     France          UEFA             2                        2.1                          0.7              0.70               0.20   
2  Argentina      CONMEBOL             1                        2.0                          0.9              0.72               0.18   
3    England          UEFA             5                        1.9                          0.8              0.68               0.22   
4      Spain          UEFA             8                        1.8                          0.6              0.65               0.25   

   loss_rate_last_20  world_cup_titles  world_cup_finals  world_cup_semifinal_appearances  squad_avg_age  squad_avg_caps  \
0                0.1                 5               

In [18]:
df.drop_duplicates(inplace=True)
print(df.isnull().sum())


team                                 0
confederation                        0
fifa_ranking                         0
avg_goals_scored_per_game            0
avg_goals_conceded_per_game          0
win_rate_last_20                     0
draw_rate_last_20                    0
loss_rate_last_20                    0
world_cup_titles                     0
world_cup_finals                     0
world_cup_semifinal_appearances      0
squad_avg_age                        0
squad_avg_caps                       0
squad_avg_club_league_tier           0
star_player_rating                   0
coach_experience_years               0
home_advantage                       0
current_form_points                  0
qualifying_goals_scored              0
qualifying_goals_conceded            0
qualifying_win_rate                  0
group_stage_opponents_avg_ranking    0
previous_wc_result                   0
odds_to_win                          0
dtype: int64


In [19]:
missing = df.isnull().sum()
if missing.sum() > 0:
    print("Missing values found, imputing:")
    print(missing[missing > 0])
    num_cols = df.select_dtypes(include=np.number).columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    for c in str_cols:
        df[c] = df[c].fillna(df[c].mode()[0])
else:
    print("No missing values found.")

No missing values found.


In [20]:
rate_check = df["win_rate_last_20"] + df["draw_rate_last_20"] + df["loss_rate_last_20"]
assert np.allclose(rate_check, 1.0, atol=0.01), "win/draw/loss rates don't sum to 1!"
print("Validated win/draw/loss rates sum to ~1.0 for every team.")

Validated win/draw/loss rates sum to ~1.0 for every team.


In [21]:
# Encode the ordinal "previous_wc_result" into a numeric score (deeper run = higher score)
result_order = {"GS": 0, "R32": 1, "R16": 2, "QF": 3, "SF": 4, "Final": 5, "W": 6}
df["previous_wc_result_score"] = df["previous_wc_result"].map(result_order)


In [29]:
# One-hot encode confederation for modelling
df_model = pd.get_dummies(df, columns=["confederation"], prefix="conf")

print("\nCleaned data preview:")
print(df[["team", "fifa_ranking", "odds_to_win", "previous_wc_result", "previous_wc_result_score"]].head())



Cleaned data preview:
        team  fifa_ranking  odds_to_win previous_wc_result  previous_wc_result_score
0     Brazil             4          5.5                 QF                         3
1     France             2          5.0                  W                         6
2  Argentina             1          6.0                  W                         6
3    England             5          7.0                 SF                         4
4      Spain             8          8.0                R16                         2


In [30]:
# ----------------------------------------------------------------------------------
# 2. FEATURE ENGINEERING — build the prediction targets
# ----------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 2: FEATURE ENGINEERING (build targets)")
print("=" * 80)

# --- Target A: Championship strength score (used to derive "winner" label) ---
# Lower FIFA ranking & odds = stronger. We invert/normalize everything to a 0-1 "goodness" scale
# then blend pedigree + current form + market odds into one composite score.

def normalize(series, invert=False):
    s = (series - series.min()) / (series.max() - series.min())
    return 1 - s if invert else s

df_model["score_ranking"]   = normalize(df_model["fifa_ranking"], invert=True)
df_model["score_odds"]      = normalize(df_model["odds_to_win"], invert=True)
df_model["score_form"]      = normalize(df_model["current_form_points"])
df_model["score_pedigree"]  = normalize(
    df_model["world_cup_titles"] * 3 + df_model["world_cup_finals"] * 1.5 +
    df_model["world_cup_semifinal_appearances"] + df_model["previous_wc_result_score"]
)
df_model["score_attack"]    = normalize(df_model["avg_goals_scored_per_game"] - df_model["avg_goals_conceded_per_game"])
df_model["score_squad"]     = normalize(df_model["star_player_rating"])

df_model["championship_strength"] = (
    0.30 * df_model["score_odds"] +
    0.20 * df_model["score_ranking"] +
    0.20 * df_model["score_pedigree"] +
    0.15 * df_model["score_form"] +
    0.10 * df_model["score_attack"] +
    0.05 * df_model["score_squad"]
)
# Classification label: is this team a top-4 "title contender"?
df_model["is_top_contender"] = (
    df_model["championship_strength"] >= df_model["championship_strength"].quantile(1 - 4/len(df_model))
).astype(int)

print("Top contenders flagged (target=1):")
print(df_model.loc[df_model["is_top_contender"] == 1, ["team", "championship_strength"]]
      .sort_values("championship_strength", ascending=False))

# --- Target B: Expected goals at the tournament (regression target) ---
# Proxy: a team typically plays ~5-7 games if they go deep. We estimate expected total
# goals using attacking rate adjusted by how far they're likely to progress (pedigree/form),
# anchored to qualifying scoring output as the most direct historical signal.
df_model["expected_games"] = 3 + 4 * df_model["championship_strength"]  # 3 (group stage) up to 7 (final)
df_model["expected_total_goals"] = (
    df_model["avg_goals_scored_per_game"] * df_model["expected_games"] * (0.85 + 0.3 * df_model["score_squad"])
)

print("\nExpected total goals (proxy target) — top 5:")
print(df_model[["team", "expected_total_goals"]].sort_values("expected_total_goals", ascending=False).head())


STEP 2: FEATURE ENGINEERING (build targets)
Top contenders flagged (target=1):
        team  championship_strength
0     Brazil               0.943583
2  Argentina               0.927527
1     France               0.885165
5    Germany               0.802595

Expected total goals (proxy target) — top 5:
        team  expected_total_goals
0     Brazil             16.926596
2  Argentina             15.433253
1     France             15.171362
6   Portugal             13.359779
3    England             12.196635


In [43]:
from sklearn.preprocessing import StandardScaler
feature_cols = [
    "fifa_ranking", "avg_goals_scored_per_game", "avg_goals_conceded_per_game",
    "win_rate_last_20", "draw_rate_last_20", "loss_rate_last_20",
    "world_cup_titles", "world_cup_finals", "world_cup_semifinal_appearances",
    "squad_avg_age", "squad_avg_caps", "squad_avg_club_league_tier",
    "star_player_rating", "coach_experience_years", "home_advantage",
    "current_form_points", "qualifying_goals_scored", "qualifying_goals_conceded",
    "qualifying_win_rate", "group_stage_opponents_avg_ranking",
    "previous_wc_result_score", "odds_to_win"
] + [c for c in df_model.columns if c.startswith("conf_")]

X = df_model[feature_cols].copy()
y_clf = df_model["is_top_contender"]
y_reg_goals = df_model["expected_total_goals"]

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

print(f"\nFeature matrix shape: {X.shape}")


# Standardize the feature matrix
scaler = StandardScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print(f"\nFeature matrix shape: {X.shape}")
print(X_scaled.head())


Feature matrix shape: (32, 27)

Feature matrix shape: (32, 27)
   fifa_ranking  avg_goals_scored_per_game  avg_goals_conceded_per_game  win_rate_last_20  draw_rate_last_20  loss_rate_last_20  \
0     -1.110551                   2.572479                    -0.984565          2.294328          -2.684914          -1.442752   
1     -1.235464                   1.886484                    -1.518566          1.663585          -0.909764          -1.442752   
2     -1.297920                   1.543487                    -0.450564          1.915882          -1.619824          -1.442752   
3     -1.048095                   1.200490                    -0.984565          1.411288          -0.199704          -1.442752   
4     -0.860726                   0.857493                    -2.052567          1.032842           0.865386          -1.442752   

   world_cup_titles  world_cup_finals  world_cup_semifinal_appearances  squad_avg_age  squad_avg_caps  squad_avg_club_league_tier  \
0          3.554

In [48]:
# =============================================================================
# MODEL 1: CLASSIFICATION
# =============================================================================

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

print("="*80)
print("Classification Model")
print("="*80)

# Leave-One-Out Cross Validation
loo = LeaveOneOut()

# Models
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=4,
    random_state=RANDOM_STATE
)

# Logistic Regression
log_reg_preds = cross_val_predict(
    log_reg,
    X_scaled,
    y_clf,
    cv=loo,
    method="predict"
)

log_reg_proba = cross_val_predict(
    log_reg,
    X_scaled,
    y_clf,
    cv=loo,
    method="predict_proba"
)[:,1]

# Random Forest
rf_preds = cross_val_predict(
    rf_clf,
    X_scaled,
    y_clf,
    cv=loo,
    method="predict"
)

rf_proba = cross_val_predict(
    rf_clf,
    X_scaled,
    y_clf,
    cv=loo,
    method="predict_proba"
)[:,1]

# Evaluation
print(f"Logistic Regression Accuracy : {accuracy_score(y_clf, log_reg_preds):.3f}")
print(f"Logistic Regression AUC      : {roc_auc_score(y_clf, log_reg_proba):.3f}")

print(f"Random Forest Accuracy       : {accuracy_score(y_clf, rf_preds):.3f}")
print(f"Random Forest AUC            : {roc_auc_score(y_clf, rf_proba):.3f}")

# Train on all data
log_reg.fit(X_scaled, y_clf)
rf_clf.fit(X_scaled, y_clf)

# Probabilities
df_model["win_proba_logreg"] = log_reg.predict_proba(X_scaled)[:,1]
df_model["win_proba_rf"] = rf_clf.predict_proba(X_scaled)[:,1]

# Blend both models
blended = (
    df_model["win_proba_logreg"] +
    df_model["win_proba_rf"]
) / 2

df_model["predicted_win_probability"] = (
    blended / blended.sum()
)

# Results table
winner_table = (
    df_model[
        ["team", "predicted_win_probability"]
    ]
    .sort_values(
        "predicted_win_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

winner_table["predicted_win_probability_pct"] = (
    winner_table["predicted_win_probability"] * 100
).round(2)

print("\nTop 10 Predicted Winners")
print(
    winner_table[
        ["team", "predicted_win_probability_pct"]
    ].head(10)
)

predicted_winner = winner_table.iloc[0]["team"]

print("\nPredicted Champion:", predicted_winner)

# Feature importance
importance = pd.Series(
    rf_clf.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\nTop 8 Important Features")
print(importance.head(8))

Classification Model
Logistic Regression Accuracy : 0.938
Logistic Regression AUC      : 0.946
Random Forest Accuracy       : 0.938
Random Forest AUC            : 0.946

Top 10 Predicted Winners
          team  predicted_win_probability_pct
0       Brazil                          24.39
1    Argentina                          23.01
2       France                          19.20
3      Germany                          19.15
4      England                           4.07
5        Spain                           3.57
6      Uruguay                           2.31
7     Portugal                           1.48
8  Netherlands                           0.72
9      Belgium                           0.68

Predicted Champion: Brazil

Top 8 Important Features
world_cup_titles                     0.157839
world_cup_semifinal_appearances      0.117352
world_cup_finals                     0.114020
odds_to_win                          0.084525
qualifying_win_rate                  0.073336
group_stage_opp

In [49]:
# ==============================================================================
# STEP 4: REGRESSION MODEL
# ==============================================================================

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.model_selection import cross_val_predict

from sklearn.metrics import mean_absolute_error, r2_score

print("=" * 80)
print("STEP 4: REGRESSION MODEL — Predict Top Goal-Scoring Team")
print("=" * 80)

# ----------------------------------------------------------------------
# Models
# ----------------------------------------------------------------------

ridge = Ridge(alpha=1.0)

rf_reg = RandomForestRegressor(
    n_estimators=300,
    max_depth=4,
    random_state=RANDOM_STATE
)

gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=RANDOM_STATE
)

models = {
    "Ridge Regression": ridge,
    "Random Forest": rf_reg,
    "Gradient Boosting": gbr
}

# ----------------------------------------------------------------------
# Evaluate using Leave-One-Out Cross Validation
# ----------------------------------------------------------------------

results = []

for name, model in models.items():

    predictions = cross_val_predict(
        model,
        X_scaled,
        y_reg_goals,
        cv=loo
    )

    mae = mean_absolute_error(
        y_reg_goals,
        predictions
    )

    r2 = r2_score(
        y_reg_goals,
        predictions
    )

    results.append([name, mae, r2])

print("\nModel Performance")

for model, mae, r2 in results:
    print(f"{model:22s} MAE = {mae:.3f}   R² = {r2:.3f}")

# ----------------------------------------------------------------------
# Train the best model
# ----------------------------------------------------------------------

gbr.fit(X_scaled, y_reg_goals)

# Predict expected goals
df_model["predicted_total_goals"] = gbr.predict(X_scaled)

# ----------------------------------------------------------------------
# Rank teams
# ----------------------------------------------------------------------

goals_table = (
    df_model[
        ["team", "predicted_total_goals"]
    ]
    .sort_values(
        by="predicted_total_goals",
        ascending=False
    )
    .reset_index(drop=True)
)

goals_table["predicted_total_goals"] = (
    goals_table["predicted_total_goals"]
    .round(2)
)

print("\nTop 10 Predicted Goal-Scoring Teams")

print(goals_table.head(10))

top_scoring_team = goals_table.iloc[0]["team"]

print("\nPredicted Top Goal-Scoring Team:", top_scoring_team)

STEP 4: REGRESSION MODEL — Predict Top Goal-Scoring Team

Model Performance
Ridge Regression       MAE = 0.420   R² = 0.977
Random Forest          MAE = 0.508   R² = 0.957
Gradient Boosting      MAE = 0.417   R² = 0.977

Top 10 Predicted Goal-Scoring Teams
          team  predicted_total_goals
0       Brazil                  16.93
1    Argentina                  15.43
2       France                  15.17
3     Portugal                  13.36
4      England                  12.20
5      Belgium                  11.39
6        Spain                  11.16
7  Netherlands                  10.76
8      Germany                  10.61
9      Uruguay                   9.36

Predicted Top Goal-Scoring Team: Brazil


In [58]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Plot 1: Win probability bar chart (top 10)
top10_win = winner_table.head(10)
axes[0, 0].barh(top10_win["team"][::-1], top10_win["predicted_win_probability_pct"][::-1], color="#2a6f97")
axes[0, 0].set_xlabel("Predicted Win Probability (%)")
axes[0, 0].set_title("Top 10 Predicted World Cup Winners", fontweight="bold")

# Plot 2: Predicted goals bar chart (top 10)
top10_goals = goals_table.head(10)
axes[0, 1].barh(top10_goals["team"][::-1], top10_goals["predicted_total_goals"][::-1], color="#bb3e03")
axes[0, 1].set_xlabel("Predicted Total Tournament Goals")
axes[0, 1].set_title("Top 10 Predicted Goal-Scoring Teams", fontweight="bold")

# Plot 3: Feature importance
top_feat = importances.head(10)
axes[1, 0].barh(top_feat.index[::-1], top_feat.values[::-1], color="#386641")
axes[1, 0].set_xlabel("Importance")
axes[1, 0].set_title("Top Features Driving Winner Prediction", fontweight="bold")

# Plot 4: FIFA ranking vs predicted win probability (sanity check / scatter)
axes[1, 1].scatter(df_model["fifa_ranking"], df_model["predicted_win_probability"] * 100,
                    s=80, c=df_model["championship_strength"], cmap="viridis", edgecolor="k")
for _, row in df_model.nlargest(6, "predicted_win_probability").iterrows():
    axes[1, 1].annotate(row["team"], (row["fifa_ranking"], row["predicted_win_probability"] * 100),
                         fontsize=8, xytext=(4, 4), textcoords="offset points")
axes[1, 1].set_xlabel("FIFA Ranking (lower = better)")
axes[1, 1].set_ylabel("Predicted Win Probability (%)")
axes[1, 1].set_title("FIFA Ranking vs. Predicted Win Probability", fontweight="bold")

plt.tight_layout()


C:\Users\HomePC\anaconda3\Lib\site-packages\PIL\Image.py:116: RuntimeWarning: The _imaging extension was built for another version of Pillow or PIL:
Core version: 12.2.0
Pillow version: 10.4.0
  # possible.


ImportError: The _imaging extension was built for another version of Pillow or PIL:
Core version: 12.2.0
Pillow version: 10.4.0